In [ ]:
"""
===============================================================================
 streamflow_shapH.py
 Daily streamflow predictability and the SHAP-based predictability horizon
 (SHAP-H) at five Polish gauging stations
 Author: Dr. Tevfik Denizhan Müftüoğlu
===============================================================================

 WHAT THE SCRIPT DOES
 --------------------
 For each station, the script builds a daily data set from GRDC discharge and
 NASA POWER meteorology, trains gradient-boosting models that forecast
 discharge h days ahead (h = 1, 2, 3, 4, 5, 6, 7, 10, 14), explains them with
 SHAP and finds the lead time at which the seasonal cycle starts to dominate
 the forecast.

 1. Data
    * The NASA POWER calendar is the master daily index; GRDC discharge is
      left-joined to it, so gaps in the discharge record do not shift lags or
      targets.
    * Predictors (38): discharge memory (Q(t), lags and rolling statistics),
      meteorological variables and their rolling sums/means, soil wetness
      (GWETTOP, GWETROOT, GWETPROF) and four day-of-year harmonics
      (doy_sin, doy_cos, doy_sin2, doy_cos2).
    * Only complete rows are used. ALLSKY_SFC_LW_DWN starts on 1988-01-01,
      so the usable record starts in 1988.
 2. Periods
      sub-training : 1988-01-01 – 2009-12-31
      validation   : 2010-01-01 – 2014-12-31   (hyperparameter tuning)
      final fit    : 1988-01-01 – 2014-12-31
      test         : 2015-01-01 – end of record (evaluation only)
 3. Models and benchmarks
    * LightGBM, tuned with Optuna (TPE sampler, 40 trials, objective =
      validation R² at h = 1); XGBoost with fixed settings.
    * Three predictor sets: Memory, Climate, Memory+Climate.
    * Benchmarks: persistence (Q(t)) and day-of-year climatology (training
      mean discharge for the calendar day of the target, 31-day smoothing).
    * Metrics on the test period: R², RMSE, MAE.
 4. SHAP and SHAP-H
    * Exact TreeSHAP on all test days (LightGBM via shap, XGBoost via its
      native pred_contribs).
    * SHAP-H (k = 3) = the shortest lead time at which any of the four
      day-of-year harmonics is among the three predictors with the largest
      mean |SHAP|.
    * Variants: k = 1 … 5, annual harmonics only, the four harmonics summed
      as one group.
    * R²-based horizons: the longest lead time up to which test R² stays
      ≥ 0.30 (and ≥ 0.50).
 5. Robustness of SHAP-H
    * five additional LightGBM random seeds;
    * XGBoost SHAP;
    * SHAP computed on training-period inputs;
    * hyperparameters tuned on the test period (comparison only; files and
      columns labelled "legacy");
    * expanding-window cross-validation, one test year at a time (2018–2023).
 6. Seasonal analysis: R² and SHAP composition for winter/spring (Dec–May)
    and summer/autumn (Jun–Nov) test days.
 7. Memory diagnostics per station: lag-1 and lag-7 autocorrelation,
    e-folding time of the de-seasonalised autocorrelation, share of discharge
    variance explained by the seasonal cycle.
 8. Figures at final print size (16 cm wide), saved as TIFF and PNG.

 HOW TO RUN (Google Colab)
 -------------------------
 Paste the script into one cell and run it. A file picker ("Choose Files")
 opens at the start; select the 10 input files below. A full run takes about
 two hours. At the end the output zip is downloaded automatically.
 The script also runs on a local computer (Jupyter / python): put the input
 files on the Desktop (or in a sub-folder of it).

   nowy_sacz__dunajec__grdc.txt      nowy_sacz__dunajec__nasa.csv
   nowy_targ__dunajec__grdc.txt      nowy_targ__dunajec__nasa.csv
   strekowa_gora__narev__grdc.txt    strekowa_gora__narev__nasa.csv
   sieradz__warta__grdc.txt          sieradz__warta__nasa.csv
   sandomierz__vistula__grdc.txt     sandomierz__vistula__nasa.csv

 OUTPUT
 ------
 Folder "shapH_outputs" and "shapH_outputs.zip" next to the input data
 (/content in Colab): all tables (CSV), figures (TIFF + PNG) and RUN_LOG.txt
 (everything printed, including package versions).
===============================================================================
"""

# ---------------------------------------------------------------------------
# 0.  Packages
# ---------------------------------------------------------------------------
import subprocess, sys
def _pip(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
for _pkg, _mod in [("optuna", "optuna"), ("lightgbm", "lightgbm"),
                   ("xgboost", "xgboost"), ("scikit-learn", "sklearn"),
                   ("shap", "shap")]:
    try:
        __import__(_mod)
    except ImportError:
        print(f"Installing {_pkg} ..."); _pip(_pkg)

import os, io, time, zipfile, warnings, platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import NullFormatter, LogLocator, FuncFormatter
import shap, lightgbm, xgboost, optuna, sklearn
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor, DMatrix
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
_T0 = time.time()

# ---------------------------------------------------------------------------
# 0b.  Output folder + log (everything printed is also written to RUN_LOG.txt)
# ---------------------------------------------------------------------------
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules or os.path.isdir("/content/sample_data")

def _desktop():
    """Desktop folder on Windows (incl. OneDrive), macOS and Linux."""
    home = os.path.expanduser("~")
    cands = [os.path.join(home, "Desktop"), os.path.join(home, "Masaüstü"),
             os.path.join(home, "OneDrive", "Desktop"), os.path.join(home, "OneDrive", "Masaüstü")]
    cands += [os.path.join(home, d, "Desktop") for d in os.listdir(home) if d.startswith("OneDrive")] if os.path.isdir(home) else []
    for c in cands:
        if os.path.isdir(c):
            return c
    return os.getcwd()

DATA_DIR = os.environ.get("SHAPH_DATA_DIR") or ("/content" if IN_COLAB else _desktop())   # input files are searched here (incl. sub-folders)
OUT = os.path.join(DATA_DIR, "shapH_outputs")  # outputs are written next to the data
os.makedirs(OUT, exist_ok=True)

class _Tee(io.TextIOBase):
    def __init__(self, *streams): self.streams = streams
    def write(self, s):
        for st in self.streams: st.write(s); st.flush()
        return len(s)
    def flush(self):
        for st in self.streams: st.flush()
_LOG = open(os.path.join(OUT, "RUN_LOG.txt"), "w", encoding="utf-8")
_ORIG_STDOUT = sys.stdout
sys.stdout = _Tee(_ORIG_STDOUT, _LOG)

def P(path):            # helper: path inside output folder
    return os.path.join(OUT, path)

# ---------------------------------------------------------------------------
# 1.  Settings
# ---------------------------------------------------------------------------
BASINS = [  # slug, display name, area (km², GRDC), grdc file, nasa file
    ("nowy_targ",  "Nowy Targ",       435, "nowy_targ__dunajec__grdc.txt",   "nowy_targ__dunajec__nasa.csv"),
    ("nowy_sacz",  "Nowy Sącz",      4337, "nowy_sacz__dunajec__grdc.txt",   "nowy_sacz__dunajec__nasa.csv"),
    ("strekowa",   "Strękowa Góra",  7199, "strekowa_gora__narev__grdc.txt", "strekowa_gora__narev__nasa.csv"),
    ("sieradz",    "Sieradz",        8156, "sieradz__warta__grdc.txt",       "sieradz__warta__nasa.csv"),
    ("sandomierz", "Sandomierz",    31809, "sandomierz__vistula__grdc.txt",  "sandomierz__vistula__nasa.csv"),
]
HORIZONS   = [1, 2, 3, 4, 5, 6, 7, 10, 14]
H_SEASONAL = [1, 3, 7]            # seasonal R² figure
H_SHAPFIG  = [1, 7]               # SHAP bar-chart figures
VAL_START, TEST_START = 2010, 2015
N_TRIALS   = 40
SEEDS      = list(range(1, 6))    # 5 additional seeds for the robustness check
CV_YEARS   = list(range(2018, 2024))
R2_THRESHOLDS = [0.30, 0.50]
K_MAIN     = 3
RNG_SEED   = 42
NJ         = -1                   # all CPU cores

# Figure style — final print size (journal: max 16 cm wide)
CM = 1 / 2.54
FIG_W = 16 * CM
plt.rcParams.update({
    "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5,
    "axes.axisbelow": True, "axes.linewidth": 0.7, "lines.linewidth": 1.3,
    "savefig.dpi": 300, "font.family": "DejaVu Sans",
})
COLORS  = {"Nowy Targ": "#c0392b", "Nowy Sącz": "#e08214", "Strękowa Góra": "#2c6fbb",
           "Sieradz": "#27ae60", "Sandomierz": "#7d3c98"}
MARKERS = {"Nowy Targ": "o", "Nowy Sącz": "s", "Strękowa Góra": "^",
           "Sieradz": "D", "Sandomierz": "v"}
C_MEM, C_MET, C_SM, C_SEAS = "#2c6fbb", "#c0392b", "#8c6d31", "#f39c12"

def logticks(axis, subs=(1, 3)):
    axis.set_major_locator(LogLocator(base=10, subs=subs, numticks=12))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    axis.set_minor_formatter(NullFormatter())

def savefig(fig, stem):
    fig.savefig(P(stem + ".png"), dpi=300, bbox_inches="tight")
    fig.savefig(P(stem + ".tif"), dpi=300, bbox_inches="tight",
                pil_kwargs={"compression": "tiff_lzw"})
    plt.close(fig)
    print(f"  saved {stem}.tif / .png")

# ---------------------------------------------------------------------------
# 2.  Data
# ---------------------------------------------------------------------------
MET_COLS = ["T2M", "T2M_MAX", "T2M_MIN", "T2MDEW", "ALLSKY_SFC_SW_DWN",
            "ALLSKY_SFC_LW_DWN", "RH2M", "PRECTOTCORR", "WS2M", "PS",
            "GWETTOP", "GWETROOT", "GWETPROF"]

def find_file(fn):
    for dp, dns, fns in os.walk(DATA_DIR):
        dns[:] = [d for d in dns if d != "shapH_outputs"]
        if fn in fns:
            return os.path.join(dp, fn)
    raise FileNotFoundError(f"'{fn}' not found in {DATA_DIR} (or its sub-folders).")

def ensure_inputs():
    """Colab: ALWAYS opens a file picker at the start so that the 10 input files can be
    selected from the computer (e.g. the Desktop). The picker re-opens until all 10
    files are present. On a local computer the files are searched on the Desktop."""
    import re as _re
    needed = [f for b in BASINS for f in b[3:5]]
    def missing():
        out = []
        for fn in needed:
            try: find_file(fn)
            except FileNotFoundError: out.append(fn)
        return out
    if IN_COLAB:
        from google.colab import files as _cf
        for attempt in range(5):
            todo = needed if attempt == 0 else missing()
            if attempt > 0 and not todo:
                break
            print("\n>>> Click 'Choose Files' below and select these files from your computer "
                  "(you can select them all at once with Ctrl/Cmd):")
            for fn in todo: print("     ", fn)
            up = _cf.upload()
            for fn, content in up.items():
                clean = _re.sub(r" \(\d+\)(?=\.[^.]+$)", "", os.path.basename(fn))   # 'x (1).txt' -> 'x.txt'
                with open(os.path.join("/content", clean), "wb") as fh:
                    fh.write(content)
            if not missing():
                break
    miss = missing()
    if miss:
        raise FileNotFoundError("Missing input files: " + ", ".join(miss))
    print("All 10 input files found:")
    for fn in needed: print("     ", find_file(fn))

def load_grdc(path):
    g = pd.read_csv(path, sep=";", comment="#", header=0, encoding="latin-1",
                    skipinitialspace=True, na_values=["-999", "-999.000", -999])
    g.columns = [c.strip() for c in g.columns]
    dcol = next(c for c in g.columns if "YYYY" in c)
    qcol = [c for c in g.columns if "Value" in c][-1]
    g["date"] = pd.to_datetime(g[dcol].astype(str).str.strip(), format="%Y-%m-%d", errors="coerce")
    g["Q"] = pd.to_numeric(g[qcol], errors="coerce")
    g.loc[g["Q"] < 0, "Q"] = np.nan
    return g[["date", "Q"]].dropna()

def load_nasa(path):
    n = pd.read_csv(path, skiprows=22, na_values=["-999", "-999.0", -999, -999.0])
    n.columns = [c.strip() for c in n.columns]
    n["date"] = pd.to_datetime(n["YEAR"].astype(int).astype(str) + "-" +
                               n["DOY"].astype(int).astype(str), format="%Y-%j")
    n = n[(n.date.dt.year >= 1984) & (n.date.dt.year <= 2025)].copy()
    for c in MET_COLS:
        n[c] = pd.to_numeric(n[c], errors="coerce")
    # isolated gaps are forward-filled; leading gaps (LW radiation before 1988) stay NaN
    n = n.set_index("date").sort_index()
    n[MET_COLS] = n[MET_COLS].ffill()
    return n.reset_index()

def build_dataset(grdc_path, nasa_path):
    g = load_grdc(grdc_path)
    n = load_nasa(nasa_path)
    g = g[(g.date >= n.date.min()) & (g.date <= n.date.max())]
    # NASA calendar = complete daily index; discharge left-joined (gaps -> NaN)
    d = n[["date"] + MET_COLS].merge(g, on="date", how="left")
    d = d[(d.date >= g.date.min()) & (d.date <= g.date.max())].sort_values("date").reset_index(drop=True)
    assert d.date.diff().dropna().eq(pd.Timedelta("1D")).all(), "calendar not continuous"
    for L in [1, 2, 3, 5, 7, 14, 30]:
        d[f"Q_lag{L}"] = d["Q"].shift(L)
    for r in [7, 14, 30]:
        d[f"Q_roll{r}"] = d["Q"].shift(1).rolling(r).mean()
    d["Q_roll7_std"]   = d["Q"].shift(1).rolling(7).std()
    d["T2M_roll7"]     = d["T2M"].rolling(7).mean()
    d["T2M_roll30"]    = d["T2M"].rolling(30).mean()
    d["posT2M_roll30"] = d["T2M"].clip(lower=0).rolling(30).sum()
    for r in [3, 7, 14, 30, 60]:
        d[f"PREC_roll{r}"] = d["PRECTOTCORR"].rolling(r).sum()
    d["snowmelt_proxy"] = d["T2M_roll7"].clip(lower=0) * d["PREC_roll30"] / (d["PREC_roll30"].mean() + 1e-9)
    doy = d["date"].dt.dayofyear
    d["doy_sin"]  = np.sin(2 * np.pi * doy / 365.25)
    d["doy_cos"]  = np.cos(2 * np.pi * doy / 365.25)
    d["doy_sin2"] = np.sin(4 * np.pi * doy / 365.25)
    d["doy_cos2"] = np.cos(4 * np.pi * doy / 365.25)
    d["season"] = np.where(d["date"].dt.month.isin([12, 1, 2, 3, 4, 5]), "WS", "SA")
    for h in HORIZONS:
        d[f"y{h}"] = d["Q"].shift(-h)
        d[f"tdate{h}"] = d["date"] + pd.Timedelta(days=h)
    return d

MEMORY  = ["Q"] + [f"Q_lag{L}" for L in [1, 2, 3, 5, 7, 14, 30]] + ["Q_roll7", "Q_roll14", "Q_roll30", "Q_roll7_std"]
SEAS    = ["doy_sin", "doy_cos", "doy_sin2", "doy_cos2"]
ANNUAL  = ["doy_sin", "doy_cos"]
METEO   = MET_COLS + ["T2M_roll7", "T2M_roll30", "posT2M_roll30", "PREC_roll3", "PREC_roll7",
                      "PREC_roll14", "PREC_roll30", "PREC_roll60", "snowmelt_proxy"]
CLIMATE = METEO + SEAS                       # "Climate" scenario = meteorology + harmonics
COMBINED = MEMORY + CLIMATE                  # 38 features
SCENARIOS = {"Memory": MEMORY, "Climate": CLIMATE, "Memory+Climate": COMBINED}
SM_FEATS = ["GWETTOP", "GWETROOT", "GWETPROF"]
assert len(MEMORY) == 12 and len(CLIMATE) == 26 and len(COMBINED) == 38

def group_of(f):
    if f in MEMORY: return "Memory"
    if f in SEAS:   return "Seasonality"
    return "Meteorological"

def label(f):           # names exactly as in the manuscript feature table
    if f == "Q": return "Q(t)"
    if f.startswith("Q_lag"): return f"Q(t−{f[5:]})"
    return f

# ---------------------------------------------------------------------------
# 3.  Models
# ---------------------------------------------------------------------------
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return dict(R2=float(r2_score(y, p)),
                RMSE=float(np.sqrt(mean_squared_error(y, p))),
                MAE=float(mean_absolute_error(y, p)))

def make_xgb(seed=RNG_SEED):
    return XGBRegressor(n_estimators=500, max_depth=7, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.8, random_state=seed, n_jobs=NJ, verbosity=0)

def make_lgb(bp, seed=RNG_SEED):
    p = dict(bp); p.update(random_state=seed, subsample_freq=1, n_jobs=NJ, verbose=-1)
    return LGBMRegressor(**p)

def optimise_lgb(Xfit, yfit, Xev, yev, n_trials=N_TRIALS):
    def obj(t):
        bp = dict(n_estimators=t.suggest_int("n_estimators", 300, 800),
                  max_depth=t.suggest_int("max_depth", 4, 9),
                  learning_rate=t.suggest_float("learning_rate", 0.01, 0.15, log=True),
                  num_leaves=t.suggest_int("num_leaves", 31, 200),
                  subsample=t.suggest_float("subsample", 0.6, 1.0),
                  colsample_bytree=t.suggest_float("colsample_bytree", 0.6, 1.0),
                  min_child_samples=t.suggest_int("min_child_samples", 10, 50),
                  reg_alpha=t.suggest_float("reg_alpha", 1e-4, 10, log=True),
                  reg_lambda=t.suggest_float("reg_lambda", 1e-4, 10, log=True))
        m = make_lgb(bp).fit(Xfit, yfit)
        return r2_score(yev, m.predict(Xev))
    st = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RNG_SEED))
    st.optimize(obj, n_trials=n_trials, show_progress_bar=False)
    return dict(st.best_params), float(st.best_value)

# ---------------------------------------------------------------------------
# 4.  SHAP helpers
# ---------------------------------------------------------------------------
def shap_lgb(model, X):
    sv = shap.TreeExplainer(model).shap_values(X)
    return np.asarray(sv[0] if isinstance(sv, list) else sv)

def shap_xgb(model, X):  # native exact TreeSHAP (avoids shap/xgboost version clashes)
    c = model.get_booster().predict(DMatrix(X), pred_contribs=True)
    return np.asarray(c)[:, :-1]          # drop bias column

def summarise(sv, feats):
    """mean |SHAP| ranking + group fractions + seasonal-rank diagnostics."""
    imp = pd.Series(np.abs(sv).mean(axis=0), index=feats).sort_values(ascending=False)
    tot = imp.sum()
    nonmem = imp[[f for f in imp.index if f not in MEMORY]].sum()
    rank = pd.Series(np.arange(1, len(imp) + 1), index=imp.index)
    best_seas = min(SEAS, key=lambda f: rank[f])
    best_ann  = min(ANNUAL, key=lambda f: rank[f])
    # grouped criterion: summed seasonal |SHAP| vs individual non-seasonal features
    seas_sum = imp[SEAS].sum()
    nonseas = imp.drop(SEAS)
    grouped_rank = int((nonseas > seas_sum).sum() + 1)
    return imp, dict(
        MemFrac=imp[MEMORY].sum() / tot,
        MetFrac=imp[[f for f in imp.index if f in METEO]].sum() / tot,
        SeasFrac=imp[SEAS].sum() / tot,
        SM_of_total=imp[SM_FEATS].sum() / tot,
        SM_of_nonmem=imp[SM_FEATS].sum() / nonmem if nonmem > 0 else np.nan,
        BestSeasRank=int(rank[best_seas]), BestSeasFeat=best_seas,
        BestAnnualRank=int(rank[best_ann]), GroupedSeasRank=grouped_rank,
        Top1=imp.index[0], Top2=imp.index[1], Top3=imp.index[2],
        Top3Vals=f"{imp.iloc[0]:.3f}|{imp.iloc[1]:.3f}|{imp.iloc[2]:.3f}",
        SeasSum=float(seas_sum), AnnualSum=float(imp[ANNUAL].sum()),
    )

def first_h(df, col, k, hcol="Horizon"):
    """first lead time at which rank column <= k; NaN if never."""
    s = df[df[col] <= k][hcol]
    return int(s.min()) if len(s) else np.nan

def fmt_h(v):
    return f"{int(v)}" if pd.notna(v) else f">{max(HORIZONS)}"

def r2_horizon(r2_by_h, thr):
    """longest lead time up to which R² stays >= thr (0 if R²(h=1) < thr)."""
    last = 0
    for h in HORIZONS:
        if r2_by_h[h] >= thr: last = h
        else: break
    return last

def climatology(train_q, train_dates):
    """smoothed day-of-year mean discharge from the training period."""
    doy = np.minimum(train_dates.dt.dayofyear.values, 365)
    m = pd.Series(train_q.values).groupby(doy).mean().reindex(range(1, 366)).interpolate()
    ext = np.concatenate([m.values[-15:], m.values, m.values[:15]])
    sm = pd.Series(ext).rolling(31, center=True).mean().values[15:-15]
    return pd.Series(sm, index=range(1, 366))

# ---------------------------------------------------------------------------
# 5.  Main loop
# ---------------------------------------------------------------------------
print(f"Data folder   : {DATA_DIR}\nOutput folder : {OUT}")
ensure_inputs()
print("=" * 72)
print(f" Python {platform.python_version()} | lightgbm {lightgbm.__version__} | xgboost {xgboost.__version__}"
      f" | shap {shap.__version__} | optuna {optuna.__version__} | sklearn {sklearn.__version__}")
print("=" * 72)

RES, SEASR2, FRAC, SEASSHAP, IMP_LONG = [], [], [], [], []
SEEDROWS, XGBROWS, LEGACY, INSAMPLE, CVROWS = [], [], [], [], []
BSTAT, PARAMS, PERIODS = [], [], []
OBSPRED = []                                           # test-period observed/predicted (h = 1, 7)

for slug, name, area, gf, nf in BASINS:
    tb = time.time()
    print(f"\n{'=' * 72}\n {name} ({area:,} km²)\n{'=' * 72}")
    d = build_dataset(find_file(gf), find_file(nf))
    base = d.dropna(subset=COMBINED)                       # complete feature rows
    first_valid = base.date.min()

    # ---- record / period bookkeeping ---------------------------------------
    tr_all = base[base.date.dt.year < TEST_START]
    PERIODS.append(dict(Basin=name, record_start=d.date.min().date(), record_end=d.date.max().date(),
                        missing_Q_days=int(d.Q.isna().sum()),
                        first_complete_feature_day=first_valid.date(),
                        train_start=tr_all.date.min().date(), train_end=tr_all.date.max().date(),
                        subtrain_end=f"{VAL_START - 1}-12-31", val=f"{VAL_START}-01-01 – {TEST_START - 1}-12-31",
                        test_start=f"{TEST_START}-01-01",
                        test_end=base[f"tdate1"][base.date.dt.year >= TEST_START].max().date()))
    print(f"  record {d.date.min().date()} – {d.date.max().date()}  | missing Q days: {int(d.Q.isna().sum())}"
          f"  | first complete feature row: {first_valid.date()}")

    # ---- memory diagnostics (training period, Q only) ----------------------
    qtr = d[(d.date >= first_valid) & (d.date.dt.year < TEST_START)][["date", "Q"]].dropna()
    clim_tr = climatology(qtr.Q, qtr.date)
    anom = qtr.Q.values - clim_tr.reindex(np.minimum(qtr.date.dt.dayofyear.values, 365)).values
    s_anom = pd.Series(anom)
    acf_e = next((L for L in range(1, 366) if s_anom.autocorr(L) < np.exp(-1)), np.nan)
    BSTAT.append(dict(Basin=name, Area_km2=area, Q_mean=d.Q.mean(), Q_max=d.Q.max(),
                      Qmax_over_Qmean=d.Q.max() / d.Q.mean(), CV_Q=d.Q.std() / d.Q.mean(),
                      ACF_lag1=pd.Series(qtr.Q.values).autocorr(1),
                      ACF_lag7=pd.Series(qtr.Q.values).autocorr(7),
                      Anom_ACF_lag7=s_anom.autocorr(7),
                      Anom_efold_days=acf_e,
                      Seasonal_var_share=1 - np.var(anom) / np.var(qtr.Q.values)))

    # ---- hyper-parameter optimisation --------------------------------------
    d1 = base.dropna(subset=["y1"])
    sub = d1[d1.date.dt.year < VAL_START]
    val = d1[(d1.date.dt.year >= VAL_START) & (d1.date.dt.year < TEST_START)]
    trn = d1[d1.date.dt.year < TEST_START]
    tst = d1[d1.date.dt.year >= TEST_START]
    print(f"  Optuna (validation protocol, {N_TRIALS} trials) ...")
    bp_val, best_val_r2 = optimise_lgb(sub[COMBINED], sub.y1, val[COMBINED], val.y1)
    print(f"    best validation R² (h=1) = {best_val_r2:.4f}")
    print(f"  Optuna (legacy protocol: test period as objective, for comparison only) ...")
    bp_leg, best_leg_r2 = optimise_lgb(trn[COMBINED], trn.y1, tst[COMBINED], tst.y1)
    print(f"    best test R² (h=1, legacy) = {best_leg_r2:.4f}")
    PARAMS.append(dict(Basin=name, protocol="validation", objective_R2=best_val_r2, **bp_val))
    PARAMS.append(dict(Basin=name, protocol="test-tuned (legacy, comparison only)", objective_R2=best_leg_r2, **bp_leg))

    # climatology for test evaluation uses the whole training period
    clim = climatology(trn.Q, trn.date)

    for h in HORIZONS:
        dh = base.dropna(subset=[f"y{h}"])
        tr = dh[dh.date.dt.year < TEST_START]
        te = dh[dh.date.dt.year >= TEST_START]
        yt = te[f"y{h}"].values

        # benchmarks
        m = metrics(yt, te.Q.values); m.update(Basin=name, Area=area, Scenario="Persistence", Model="-", Horizon=h); RES.append(m)
        cp = clim.reindex(np.minimum(te[f"tdate{h}"].dt.dayofyear.values, 365)).values
        m = metrics(yt, cp); m.update(Basin=name, Area=area, Scenario="Climatology", Model="-", Horizon=h); RES.append(m)

        for sc, feats in SCENARIOS.items():
            for lbl in ["XGB", "LGB"]:
                mdl = make_xgb() if lbl == "XGB" else make_lgb(bp_val)
                mdl.fit(tr[feats], tr[f"y{h}"])
                pred = mdl.predict(te[feats])
                m = metrics(yt, pred); m.update(Basin=name, Area=area, Scenario=sc, Model=lbl, Horizon=h); RES.append(m)
                if sc != "Memory+Climate":
                    continue
                if lbl == "XGB":
                    _, sx = summarise(shap_xgb(mdl, te[feats]), feats)
                    XGBROWS.append(dict(Basin=name, Horizon=h, R2=m["R2"], **sx))
                    continue
                # ---- main model: LightGBM Memory+Climate -------------------
                lgb_main, pred_main = mdl, pred
                if h in (1, 7):
                    OBSPRED.append(pd.DataFrame(dict(Basin=name, Horizon=h, Date=te[f"tdate{h}"].dt.date.values,
                                                     Observed=yt, Predicted=pred)))
                sv = shap_lgb(mdl, te[feats])
                imp, s = summarise(sv, feats)
                FRAC.append(dict(Basin=name, Area=area, Horizon=h, R2=m["R2"], **s))
                for rk, (f, v) in enumerate(imp.items(), 1):
                    IMP_LONG.append(dict(Basin=name, Horizon=h, Rank=rk, Feature=f, Group=group_of(f), MeanAbsSHAP=v))
                # seasonal R² and seasonal SHAP
                for sn in ["WS", "SA"]:
                    msk = (te.season == sn).values
                    _, ss = summarise(sv[msk], feats)
                    r2s = r2_score(yt[msk], pred[msk])
                    r2p = r2_score(yt[msk], te.Q.values[msk])
                    SEASR2.append(dict(Basin=name, Season=sn, Horizon=h, R2=r2s, R2_persistence=r2p, N=int(msk.sum())))
                    SEASSHAP.append(dict(Basin=name, Season=sn, Horizon=h, R2=r2s, **ss))
                # SHAP on training-period inputs (no verification data needed)
                rs = np.random.default_rng(RNG_SEED)
                idx = rs.choice(len(tr), size=min(4000, len(tr)), replace=False)
                _, si = summarise(shap_lgb(mdl, tr[feats].iloc[idx]), feats)
                INSAMPLE.append(dict(Basin=name, Horizon=h, **si))

        # ---- legacy protocol: hyperparameters tuned on the test period (comparison only)
        ml = make_lgb(bp_leg).fit(tr[COMBINED], tr[f"y{h}"])
        pl = ml.predict(te[COMBINED])
        _, sl = summarise(shap_lgb(ml, te[COMBINED]), COMBINED)
        LEGACY.append(dict(Basin=name, Horizon=h, R2_legacy=r2_score(yt, pl), **{k + "_legacy": v for k, v in sl.items()}))

        # ---- seed robustness -------------------------------------------------
        for sd in SEEDS:
            ms = make_lgb(bp_val, seed=sd).fit(tr[COMBINED], tr[f"y{h}"])
            _, ss = summarise(shap_lgb(ms, te[COMBINED]), COMBINED)
            SEEDROWS.append(dict(Basin=name, Horizon=h, Seed=sd, R2=r2_score(yt, ms.predict(te[COMBINED])),
                                 BestSeasRank=ss["BestSeasRank"], BestAnnualRank=ss["BestAnnualRank"],
                                 GroupedSeasRank=ss["GroupedSeasRank"], SeasFrac=ss["SeasFrac"], MemFrac=ss["MemFrac"]))

        r2_main = [r for r in RES if r["Basin"] == name and r["Horizon"] == h and r["Scenario"] == "Memory+Climate" and r["Model"] == "LGB"][0]["R2"]
        fr = FRAC[-1]
        print(f"   h={h:>2}  R²={r2_main:6.3f}  mem={fr['MemFrac']*100:5.1f}%  seas={fr['SeasFrac']*100:5.1f}%"
              f"  best-harmonic rank={fr['BestSeasRank']:>2} ({fr['BestSeasFeat']})  top3: {fr['Top1']}, {fr['Top2']}, {fr['Top3']}")

    # ---- expanding-window CV: R² and SHAP-H inside each fold ---------------
    print("  expanding-window CV (2018–2023) ...")
    for ty in CV_YEARS:
        for h in HORIZONS:
            dh = base.dropna(subset=[f"y{h}"])
            tr = dh[dh.date.dt.year < ty]; te = dh[dh.date.dt.year == ty]
            if len(te) < 100:
                continue
            mc = make_lgb(bp_val).fit(tr[COMBINED], tr[f"y{h}"])
            pc = mc.predict(te[COMBINED])
            _, sc_ = summarise(shap_lgb(mc, te[COMBINED]), COMBINED)
            CVROWS.append(dict(Basin=name, Year=ty, Horizon=h, R2=r2_score(te[f"y{h}"], pc),
                               BestSeasRank=sc_["BestSeasRank"], MemFrac=sc_["MemFrac"], SeasFrac=sc_["SeasFrac"]))
    print(f"  {name} done in {(time.time() - tb) / 60:.1f} min")

# ---------------------------------------------------------------------------
# 6.  Tables
# ---------------------------------------------------------------------------
RES = pd.DataFrame(RES); FRAC = pd.DataFrame(FRAC); SEASR2 = pd.DataFrame(SEASR2)
SEASSHAP = pd.DataFrame(SEASSHAP); IMP = pd.DataFrame(IMP_LONG); SEEDS_DF = pd.DataFrame(SEEDROWS)
XGB_DF = pd.DataFrame(XGBROWS); LEG = pd.DataFrame(LEGACY); INS = pd.DataFrame(INSAMPLE)
CV = pd.DataFrame(CVROWS); BST = pd.DataFrame(BSTAT); PAR = pd.DataFrame(PARAMS); PER = pd.DataFrame(PERIODS)
NAMES = [b[1] for b in BASINS]

for df, fn in [(RES, "results_all_models.csv"), (FRAC, "shap_fractions_main.csv"),
               (SEASR2, "seasonal_r2.csv"), (SEASSHAP, "shap_fractions_seasonal.csv"),
               (IMP, "shap_importance_long.csv"), (SEEDS_DF, "robustness_seeds.csv"),
               (XGB_DF, "robustness_xgboost_shap.csv"), (LEG, "legacy_r0_protocol.csv"),
               (INS, "robustness_insample_shap.csv"), (CV, "cv_expanding_window.csv"),
               (BST, "basin_memory_diagnostics.csv"), (PAR, "hyperparameters.csv"),
               (PER, "data_periods.csv")]:
    df.to_csv(P(fn), index=False)

def r2_of(name, h, sc="Memory+Climate", model="LGB", df=RES):
    return float(df[(df.Basin == name) & (df.Horizon == h) & (df.Scenario == sc) & (df.Model == model)].R2.iloc[0])

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

print("\n\n" + "#" * 72 + "\n#  RESULTS FOR THE MANUSCRIPT\n" + "#" * 72)
print("\n[T0] Data periods actually used"); print(PER.to_string(index=False))
print("\n[T0b] Tuned hyper-parameters"); print(PAR.round(4).to_string(index=False))
print("\n[T1] Memory diagnostics (training period)"); print(BST.round(3).to_string(index=False))

# R² table (main) with benchmarks
tab = []
for nm in NAMES:
    row = {"Basin": nm}
    for h in HORIZONS: row[f"h{h}"] = r2_of(nm, h)
    row["RMSE_h1"] = float(RES[(RES.Basin == nm) & (RES.Horizon == 1) & (RES.Scenario == "Memory+Climate") & (RES.Model == "LGB")].RMSE.iloc[0])
    tab.append(row)
TAB2 = pd.DataFrame(tab); TAB2.to_csv(P("table_r2_main.csv"), index=False)
print("\n[T2] Test R², Memory+Climate LightGBM"); print(TAB2.round(3).to_string(index=False))

for sc in ["Persistence", "Climatology"]:
    t = RES[RES.Scenario == sc].pivot(index="Basin", columns="Horizon", values="R2").reindex(NAMES)
    print(f"\n[T2-{sc}] test R² of the {sc.lower()} benchmark"); print(t.round(3).to_string())
    t.to_csv(P(f"table_r2_{sc.lower()}.csv"))

# Scenario comparison h=1
t3 = []
for nm in NAMES:
    t3.append(dict(Basin=nm, Persistence=r2_of(nm, 1, "Persistence", "-"), Climatology=r2_of(nm, 1, "Climatology", "-"),
                   Memory=r2_of(nm, 1, "Memory"), Climate=r2_of(nm, 1, "Climate"), Combined=r2_of(nm, 1),
                   dR2_vs_persistence=r2_of(nm, 1) - r2_of(nm, 1, "Persistence", "-")))
T3 = pd.DataFrame(t3); T3.to_csv(P("table_scenarios_h1.csv"), index=False)
print("\n[T3] Scenario comparison at h=1 (LightGBM)"); print(T3.round(3).to_string(index=False))

# XGB vs LGB difference
dx = []
for nm in NAMES:
    for h in HORIZONS:
        for sc in SCENARIOS:
            dx.append(abs(r2_of(nm, h, sc, "XGB") - r2_of(nm, h, sc, "LGB")))
print(f"\n[XGB vs LGB] max |ΔR²| over all basins/horizons/scenarios = {max(dx):.3f}; "
      f"median = {np.median(dx):.3f}")
dxc = [abs(r2_of(nm, h, "Memory+Climate", "XGB") - r2_of(nm, h)) for nm in NAMES for h in HORIZONS]
print(f"[XGB vs LGB] Memory+Climate only: max |ΔR²| = {max(dxc):.3f}")

# SHAP fractions table
T4 = FRAC.pivot(index="Basin", columns="Horizon", values="MemFrac").reindex(NAMES) * 100
print("\n[T4a] Memory SHAP fraction (%)"); print(T4.round(1).to_string())
T4b = FRAC.pivot(index="Basin", columns="Horizon", values="SeasFrac").reindex(NAMES) * 100
print("\n[T4b] Seasonality SHAP fraction (%), all four harmonics"); print(T4b.round(1).to_string())
T4c = FRAC.pivot(index="Basin", columns="Horizon", values="BestSeasRank").reindex(NAMES)
print("\n[T4c] Rank of the best-ranked seasonal harmonic"); print(T4c.to_string())
T4d = FRAC.pivot(index="Basin", columns="Horizon", values="SM_of_nonmem").reindex(NAMES) * 100
print("\n[T4d] Soil-moisture (GWETTOP+GWETROOT+GWETPROF) share of non-memory SHAP (%)"); print(T4d.round(1).to_string())
T4.to_csv(P("table_memfrac.csv")); T4b.to_csv(P("table_seasfrac.csv")); T4c.to_csv(P("table_best_harmonic_rank.csv"))

print("\n[T4e] Top-3 SHAP features (mean |SHAP|, m³/s) per basin and lead time")
for nm in NAMES:
    print(f"  {nm}")
    for _, r in FRAC[FRAC.Basin == nm].iterrows():
        v = r.Top3Vals.split("|")
        print(f"    h={r.Horizon:>2}: {r.Top1} ({v[0]}) | {r.Top2} ({v[1]}) | {r.Top3} ({v[2]})"
              f"   [sum doy_sin+doy_cos = {r.AnnualSum:.3f}; all four harmonics = {r.SeasSum:.3f}]")

# ---- SHAP-H summary ---------------------------------------------------------
summ = []
for nm in NAMES:
    f = FRAC[FRAC.Basin == nm]
    r2h = {h: r2_of(nm, h) for h in HORIZONS}
    row = dict(Basin=nm)
    for thr in R2_THRESHOLDS:
        row[f"H_R2_{thr:.2f}"] = r2_horizon(r2h, thr)
    # first lead time at which the model is no better than climatology
    beat = [h for h in HORIZONS if r2h[h] <= r2_of(nm, h, "Climatology", "-")]
    row["H_model_le_climatology"] = beat[0] if beat else np.nan
    for k in [1, 2, 3, 4, 5]:
        row[f"SHAPH_k{k}"] = first_h(f, "BestSeasRank", k)
    row["SHAPH_annual_k3"] = first_h(f, "BestAnnualRank", K_MAIN)
    row["SHAPH_grouped_k3"] = first_h(f, "GroupedSeasRank", K_MAIN)
    row["SHAPH_legacyR0_k3"] = first_h(LEG[LEG.Basin == nm], "BestSeasRank_legacy", K_MAIN)
    row["SHAPH_xgb_k3"] = first_h(XGB_DF[XGB_DF.Basin == nm], "BestSeasRank", K_MAIN)
    row["SHAPH_trainingSHAP_k3"] = first_h(INS[INS.Basin == nm], "BestSeasRank", K_MAIN)
    sh = [first_h(SEEDS_DF[(SEEDS_DF.Basin == nm) & (SEEDS_DF.Seed == s)], "BestSeasRank", K_MAIN) for s in SEEDS]
    shv = pd.Series(sh)
    row["SHAPH_seeds_mode"] = fmt_h(shv.mode(dropna=False).iloc[0])
    row["SHAPH_seeds_agree_with_main"] = int(sum((pd.isna(x) and pd.isna(row['SHAPH_k3'])) or x == row["SHAPH_k3"] for x in sh))
    row["SHAPH_seeds_values"] = ",".join(fmt_h(x) for x in sh)
    if pd.notna(row["SHAPH_k3"]):
        hh = int(row["SHAPH_k3"])
        row["R2_at_SHAPH"] = r2h[hh]
        row["R2clim_at_SHAPH"] = r2_of(nm, hh, "Climatology", "-")
    summ.append(row)
SHAPH = pd.DataFrame(summ); SHAPH.to_csv(P("table_shapH_summary.csv"), index=False)
print("\n[T5] SHAP-H and R²-based horizons (main k = 3; other columns = sensitivity/robustness)")
print(SHAPH.to_string(index=False))

# legacy (test-tuned) vs validation-tuned comparison
lv = []
for nm in NAMES:
    for h in HORIZONS:
        lv.append(dict(Basin=nm, Horizon=h, R2_validation_tuned=r2_of(nm, h),
                       R2_test_tuned_R0=float(LEG[(LEG.Basin == nm) & (LEG.Horizon == h)].R2_legacy.iloc[0])))
LV = pd.DataFrame(lv); LV["dR2"] = LV.R2_validation_tuned - LV.R2_test_tuned_R0
LV.to_csv(P("legacy_vs_validation.csv"), index=False)
print("\n[T6] Effect of the tuning protocol on test R² (validation-tuned minus test-tuned)")
print(LV.pivot(index="Basin", columns="Horizon", values="dR2").reindex(NAMES).round(3).to_string())
print(LV[LV.Horizon == 1].round(4).to_string(index=False))

# seasonal
print("\n[T7] Seasonal R² (WS = Dec–May, SA = Jun–Nov), Memory+Climate LightGBM")
print(SEASR2.pivot_table(index=["Basin", "Season"], columns="Horizon", values="R2").round(3).to_string())
print("\n[T7b] Seasonal SHAP: memory / meteorological / seasonality fractions and soil-moisture shares at h = 1, 3, 7, 14")
cols = ["Basin", "Season", "Horizon", "R2", "MemFrac", "MetFrac", "SeasFrac", "SM_of_total", "SM_of_nonmem", "BestSeasRank", "Top1", "Top2", "Top3"]
print(SEASSHAP[SEASSHAP.Horizon.isin([1, 3, 7, 14])][cols].round(3).to_string(index=False))

# CV
for h in [1, 3, 7]:
    t = CV[CV.Horizon == h].pivot(index="Basin", columns="Year", values="R2").reindex(NAMES)
    t["Mean"] = t.mean(axis=1); t["SD"] = t.iloc[:, :len(CV_YEARS)].std(axis=1)
    t.to_csv(P(f"table_cv_h{h}.csv"))
    print(f"\n[T8] Expanding-window CV R², h={h}"); print(t.round(3).to_string())
cvh = []
for nm in NAMES:
    for ty in CV_YEARS:
        f = CV[(CV.Basin == nm) & (CV.Year == ty)]
        cvh.append(dict(Basin=nm, Year=ty, SHAPH_k3=fmt_h(first_h(f, "BestSeasRank", K_MAIN)),
                        H_R2_030=r2_horizon({h: float(f[f.Horizon == h].R2.iloc[0]) for h in HORIZONS}, 0.30)))
CVH = pd.DataFrame(cvh); CVH.to_csv(P("table_cv_shapH.csv"), index=False)
print("\n[T9] SHAP-H (k=3) and R²≥0.30 horizon inside each CV fold")
print(CVH.pivot(index="Basin", columns="Year", values="SHAPH_k3").reindex(NAMES).to_string())
print(CVH.pivot(index="Basin", columns="Year", values="H_R2_030").reindex(NAMES).to_string())

# agreement between SHAP-H(k) and the R²-based horizon (rank agreement across basins)
print("\n[T10] Spearman rank correlation across the five basins between SHAP-H(k) and H_R2(0.30)")
cap = max(HORIZONS) + 1
for k in [1, 2, 3, 4, 5]:
    a = SHAPH[f"SHAPH_k{k}"].fillna(cap); b = SHAPH["H_R2_0.30"]
    print(f"   k={k}: rho = {a.rank().corr(b.rank()):.3f}   values: {list(SHAPH[f'SHAPH_k{k}'].map(fmt_h))}")

# ---------------------------------------------------------------------------
# 7.  Figures (16 cm wide, >= 7 pt), drawn from the saved CSV tables
# ---------------------------------------------------------------------------
LET = "abcdef"
NAMES = [b[1] for b in BASINS]
AREAS = [b[2] for b in BASINS]

def _minus(v, fmt):
    """number with a typographic minus sign."""
    return format(v, fmt).replace("-", "−")

def grid_axes(nrow=2, ncol=3, h_cm=11.5, sharey=False):
    fig, axes = plt.subplots(nrow, ncol, figsize=(FIG_W, h_cm * CM), sharey=sharey)
    return fig, axes.ravel()

def auto_logticks(axis, lo, hi):
    """log-axis tick density chosen from the number of decades shown."""
    dec = np.log10(hi / lo)
    logticks(axis, subs=(1, 2, 5) if dec < 1.2 else (1, 3) if dec < 2.2 else (1.0,))

def draw_figures():
    rd = lambda fn: pd.read_csv(P(fn))
    RES_, FRAC_, SHAPH_ = rd("results_all_models.csv"), rd("shap_fractions_main.csv"), rd("table_shapH_summary.csv")
    IMP_, SEASR2_, SEEDS_ = rd("shap_importance_long.csv"), rd("seasonal_r2.csv"), rd("robustness_seeds.csv")
    SEASSHAP_ = rd("shap_fractions_seasonal.csv")
    cap = max(HORIZONS) + 1

    def r2(nm, h, sc="Memory+Climate", model="LGB"):
        return float(RES_[(RES_.Basin == nm) & (RES_.Horizon == h) & (RES_.Scenario == sc) & (RES_.Model == model)].R2.iloc[0])

    # Fig. 1 — observed vs predicted (log–log), h = 1 and h = 7
    OP = pd.concat(OBSPRED, ignore_index=True)       # observed/predicted test values (h = 1, 7)
    fig, ax = grid_axes(h_cm=11.5)
    for i, nm in enumerate(NAMES):
        a = ax[i]
        for h, c in [(7, "#e08214"), (1, "#2c6fbb")]:
            f = OP[(OP.Basin == nm) & (OP.Horizon == h)]
            a.scatter(f.Observed, np.clip(f.Predicted, 1e-3, None), s=2, alpha=0.25, color=c, rasterized=True, lw=0)
        o1 = OP[(OP.Basin == nm) & (OP.Horizon == 1)].Observed
        lo, hi = max(o1.min() * 0.8, 1e-2), o1.max() * 1.25
        a.plot([lo, hi], [lo, hi], "k-", lw=0.8)
        a.set_xscale("log"); a.set_yscale("log"); a.set_xlim(lo, hi); a.set_ylim(lo, hi)
        logticks(a.xaxis); logticks(a.yaxis)
        a.set_aspect("equal", adjustable="box")
        a.set_title(f"({LET[i]}) {nm}", loc="left")
        a.text(0.97, 0.05, f"R² h=1: {_minus(r2(nm, 1), '.3f')}\nR² h=7: {_minus(r2(nm, 7), '.3f')}",
               transform=a.transAxes, ha="right", va="bottom", fontsize=7,
               bbox=dict(fc="white", ec="0.7", lw=0.5, alpha=0.9))
        if i % 3 == 0: a.set_ylabel("Predicted Q (m³/s)")
        if i >= 2: a.set_xlabel("Observed Q (m³/s)")
    ax[5].axis("off")
    ax[5].legend(handles=[Line2D([0], [0], marker="o", ls="", color="#2c6fbb", label="h = 1 day"),
                          Line2D([0], [0], marker="o", ls="", color="#e08214", label="h = 7 days"),
                          Line2D([0], [0], color="k", lw=0.8, label="1:1 line")], loc="center", frameon=False)
    fig.tight_layout(); savefig(fig, "Fig_obs_vs_pred")

    # Fig. 2 — skill decay: R² (model and persistence) and RMSE
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(FIG_W, 7.0 * CM))
    for nm in NAMES:
        c, mk = COLORS[nm], MARKERS[nm]
        a1.plot(HORIZONS, [r2(nm, h) for h in HORIZONS], marker=mk, ms=3.5, color=c, label=nm)
        a1.plot(HORIZONS, [r2(nm, h, "Persistence", "-") for h in HORIZONS], ls="--", lw=0.9, color=c)
        a2.plot(HORIZONS, [float(RES_[(RES_.Basin == nm) & (RES_.Horizon == h) & (RES_.Scenario == "Memory+Climate")
                                      & (RES_.Model == "LGB")].RMSE.iloc[0]) for h in HORIZONS], marker=mk, ms=3.5, color=c)
    a1.axhline(0.30, color="0.4", lw=0.7, ls=":"); a1.axhline(0, color="k", lw=0.6)
    a1.set_ylim(max(-1.0, a1.get_ylim()[0]), 1.02)
    a1.set_xlabel("Lead time h (days)"); a1.set_ylabel("Test R²"); a1.set_title("(a) R²", loc="left")
    a2.set_yscale("log"); logticks(a2.yaxis); a2.set_xlabel("Lead time h (days)"); a2.set_ylabel("Test RMSE (m³/s)")
    a2.set_title("(b) RMSE", loc="left")
    for a in (a1, a2): a.set_xticks(HORIZONS)
    hd = [Line2D([0], [0], color=COLORS[n], marker=MARKERS[n], ms=3.5, label=n) for n in NAMES]
    hd += [Line2D([0], [0], color="0.3", ls="--", lw=0.9, label="Persistence"),
           Line2D([0], [0], color="0.4", ls=":", lw=0.7, label="R² = 0.30")]
    fig.legend(handles=hd, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.13), frameon=False)
    fig.tight_layout(); savefig(fig, "Fig_skill_decay")

    # Fig. 3 — area vs one-day skill and area vs horizons (R² horizon and SHAP-H)
    y_cap = max(HORIZONS) + 2.5                     # plotting position of "not reached within 14 days"
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(FIG_W, 6.8 * CM))
    for nm, ar in zip(NAMES, AREAS):
        c, mk = COLORS[nm], MARKERS[nm]
        a1.scatter(ar, r2(nm, 1), color=c, marker=mk, s=28, zorder=3)
        s = SHAPH_[SHAPH_.Basin == nm].iloc[0]
        shv = s["SHAPH_k3"] if pd.notna(s["SHAPH_k3"]) else y_cap
        a2.plot([ar, ar], [s["H_R2_0.30"], shv], color=c, lw=0.7, zorder=2)
        a2.scatter(ar, s["H_R2_0.30"], color=c, marker=mk, s=28, zorder=3)
        a2.scatter(ar, shv, facecolor="white", edgecolor=c, marker=mk, s=28, zorder=4, lw=1.1)
    for a in (a1, a2): a.set_xscale("log"); a.set_xlabel("Catchment area (km², log scale)")
    a1.set_ylabel("Test R² at h = 1 day"); a1.set_title("(a) One-day skill", loc="left")
    a2.set_ylabel("Lead time (days)"); a2.set_title("(b) Horizons", loc="left")
    a2.set_ylim(-0.5, y_cap + 1.2)
    a2.set_yticks([0] + HORIZONS + [y_cap]); a2.set_yticklabels(["0"] + [str(h) for h in HORIZONS] + [f">{max(HORIZONS)}"])
    a2.axhspan(max(HORIZONS) + 1, y_cap + 1.2, color="0.93", lw=0, zorder=0)
    hd = [Line2D([0], [0], color=COLORS[n], marker=MARKERS[n], ls="", label=n) for n in NAMES]
    hd += [Line2D([0], [0], marker="o", ls="", color="0.3", label="R²-based horizon (R² ≥ 0.30)"),
           Line2D([0], [0], marker="o", ls="", mfc="white", mec="0.3", label="SHAP-H (k = 3)")]
    fig.legend(handles=hd, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.17), frameon=False)
    fig.tight_layout(); savefig(fig, "Fig_area_skill_horizon")

    # Fig. 4 — SHAP group fractions vs lead time, with SHAP-H and R² horizon
    fig, ax = grid_axes(h_cm=11.0, sharey=True)
    for i, nm in enumerate(NAMES):
        a = ax[i]; f = FRAC_[FRAC_.Basin == nm].sort_values("Horizon")
        a.plot(f.Horizon, f.MemFrac * 100, "o-", ms=3, color=C_MEM)
        a.plot(f.Horizon, f.MetFrac * 100, "s-", ms=3, color=C_MET)
        a.plot(f.Horizon, f.SeasFrac * 100, "^-", ms=3, color=C_SEAS)
        s = SHAPH_[SHAPH_.Basin == nm].iloc[0]
        if pd.notna(s["SHAPH_k3"]): a.axvline(s["SHAPH_k3"], color="0.25", ls="--", lw=0.9)
        if s["H_R2_0.30"] > 0: a.axvline(s["H_R2_0.30"], color="0.55", ls=":", lw=1.0)
        a.set_title(f"({LET[i]}) {nm}", loc="left"); a.set_ylim(0, 100); a.set_xticks(HORIZONS)
        a.tick_params(axis="x", labelsize=6.5)
        if i % 3 == 0: a.set_ylabel("Share of total mean |SHAP| (%)")
        if i >= 2: a.set_xlabel("Lead time h (days)")
    ax[5].axis("off")
    ax[5].legend(handles=[Line2D([0], [0], color=C_MEM, marker="o", ms=3, label="Discharge memory"),
                          Line2D([0], [0], color=C_MET, marker="s", ms=3, label="Meteorological"),
                          Line2D([0], [0], color=C_SEAS, marker="^", ms=3, label="Seasonality (4 harmonics)"),
                          Line2D([0], [0], color="0.25", ls="--", label="SHAP-H (k = 3)"),
                          Line2D([0], [0], color="0.55", ls=":", label="Last h with R² ≥ 0.30")],
                 loc="center", frameon=False)
    fig.tight_layout(); savefig(fig, "Fig_shap_fractions")

    # Fig. 5/6 — SHAP bar charts at h = 1 and h = 7 (ten largest mean |SHAP|)
    for h in H_SHAPFIG:
        fig, ax = grid_axes(h_cm=12.5)
        for i, nm in enumerate(NAMES):
            a = ax[i]
            t = IMP_[(IMP_.Basin == nm) & (IMP_.Horizon == h)].sort_values("Rank").head(10)[::-1]
            cols_ = [C_MEM if g == "Memory" else C_SEAS if g == "Seasonality" else C_MET for g in t.Group]
            a.barh(range(len(t)), t.MeanAbsSHAP, color=cols_, alpha=0.9, height=0.7)
            a.set_yticks(range(len(t))); a.set_yticklabels([label(x) for x in t.Feature], fontsize=7)
            a.set_xscale("log")
            lo, hi = t.MeanAbsSHAP.min() * 0.7, t.MeanAbsSHAP.max() * 1.25
            a.set_xlim(lo, hi); auto_logticks(a.xaxis, lo, hi)
            a.set_title(f"({LET[i]}) {nm}", loc="left")
            a.set_xlabel("Mean |SHAP| (m³/s, log scale)", fontsize=7); a.tick_params(axis="x", labelsize=6.5)
        ax[5].axis("off")
        ax[5].legend(handles=[Patch(color=C_MEM, label="Discharge memory"), Patch(color=C_MET, label="Meteorological"),
                              Patch(color=C_SEAS, label="Seasonality (doy harmonics)")], loc="center", frameon=False)
        fig.tight_layout(); savefig(fig, f"Fig_shap_bars_h{h}")

    # Fig. 7 — rank of the best-ranked seasonal harmonic vs lead time; range over seeds shaded
    fig, ax = grid_axes(h_cm=12.5, sharey=True)
    for i, nm in enumerate(NAMES):
        a = ax[i]; f = FRAC_[FRAC_.Basin == nm].sort_values("Horizon")
        sd = SEEDS_[SEEDS_.Basin == nm].groupby("Horizon").BestSeasRank.agg(["min", "max"]).reindex(HORIZONS)
        a.fill_between(HORIZONS, sd["min"], sd["max"], color=COLORS[nm], alpha=0.2, lw=0)
        a.plot(f.Horizon, f.BestSeasRank, "o-", ms=3, color=COLORS[nm])
        for k, ls in [(2, ":"), (3, "--"), (4, ":")]:
            a.axhline(k + 0.5, color="0.35" if k == 3 else "0.65", ls=ls, lw=0.8)
        a.set_yscale("log"); a.set_ylim(40, 0.8)
        a.set_yticks([1, 2, 3, 4, 5, 10, 20, 38]); a.set_yticklabels(["1", "2", "3", "4", "5", "10", "20", "38"])
        a.yaxis.set_minor_locator(plt.NullLocator()); a.tick_params(axis="y", labelsize=7)
        a.set_xticks(HORIZONS); a.tick_params(axis="x", labelsize=6.5)
        a.set_title(f"({LET[i]}) {nm}", loc="left")
        if i >= 2: a.set_xlabel("Lead time h (days)")
    ax[5].axis("off")
    ax[5].legend(handles=[Line2D([0], [0], color="0.3", marker="o", ms=3, label=f"Main model (seed {RNG_SEED})"),
                          Patch(color="0.3", alpha=0.2, label=f"Range over {len(SEEDS)} seeds"),
                          Line2D([0], [0], color="0.35", ls="--", label="Top-3 boundary (k = 3)"),
                          Line2D([0], [0], color="0.65", ls=":", label="Top-2 / top-4 boundaries")],
                 loc="center", frameon=False)
    fig.supylabel("SHAP rank of the best-ranked seasonal harmonic (log scale)", fontsize=8)
    fig.tight_layout(); savefig(fig, "Fig_harmonic_rank")

    # Fig. 8 — seasonal R² (horizontal bars; station names written once, on the left)
    fig, ax = plt.subplots(1, 3, figsize=(FIG_W, 6.5 * CM), sharey=True, sharex=True)
    y = np.arange(len(NAMES)); w = 0.38
    vals = []
    for j, h in enumerate(H_SEASONAL):
        a = ax[j]
        ws = [float(SEASR2_[(SEASR2_.Basin == n) & (SEASR2_.Season == "WS") & (SEASR2_.Horizon == h)].R2.iloc[0]) for n in NAMES]
        sa = [float(SEASR2_[(SEASR2_.Basin == n) & (SEASR2_.Season == "SA") & (SEASR2_.Horizon == h)].R2.iloc[0]) for n in NAMES]
        vals += ws + sa
        a.barh(y - w / 2, ws, w, color="#2c6fbb")
        a.barh(y + w / 2, sa, w, color="#e08214")
        a.axvline(0, color="k", lw=0.6)
        a.set_title(f"({LET[j]}) h = {h} day{'s' if h > 1 else ''}", loc="left")
        a.set_xlabel("Test R²")
        a.grid(axis="y", visible=False)
    ax[0].set_yticks(y); ax[0].set_yticklabels(NAMES); ax[0].invert_yaxis()
    ax[0].set_xlim(min(-0.1, np.floor(min(vals) * 10) / 10 - 0.05), 1.05)
    ax[0].xaxis.set_major_locator(matplotlib.ticker.MultipleLocator(0.5))
    ax[0].xaxis.set_minor_locator(matplotlib.ticker.MultipleLocator(0.25))
    fig.legend(handles=[Patch(color="#2c6fbb", label="Winter/spring (Dec–May)"),
                        Patch(color="#e08214", label="Summer/autumn (Jun–Nov)")],
               loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.08), frameon=False)
    fig.tight_layout(); savefig(fig, "Fig_seasonal_r2")

    # Fig. 9 — seasonal SHAP composition at h = 7 (stacked bars, WS vs SA)
    fig, a = plt.subplots(figsize=(FIG_W, 6.5 * CM))
    xl, xs = [], []
    for i, nm in enumerate(NAMES):
        for j, sn in enumerate(["WS", "SA"]):
            r = SEASSHAP_[(SEASSHAP_.Basin == nm) & (SEASSHAP_.Season == sn) & (SEASSHAP_.Horizon == 7)].iloc[0]
            x = i * 2.6 + j
            parts = [r.MemFrac, r.MetFrac - r.SM_of_total, r.SM_of_total, r.SeasFrac]
            btm = 0
            for v, c in zip(parts, [C_MEM, C_MET, C_SM, C_SEAS]):
                a.bar(x, v * 100, 0.85, bottom=btm, color=c); btm += v * 100
            a.text(x, 101, f"R²={_minus(r.R2, '.2f')}", ha="center", va="bottom", fontsize=6, rotation=90)
            xl.append(x); xs.append(sn)
    for i, nm in enumerate(NAMES):
        a.text(i * 2.6 + 0.5, -16, nm, ha="center", va="top", fontsize=7.5)
    a.set_xticks(xl); a.set_xticklabels(xs, fontsize=7); a.set_ylim(0, 132); a.set_yticks(range(0, 101, 20))
    a.set_ylabel("Share of mean |SHAP| at h = 7 (%)")
    fig.legend(handles=[Patch(color=C_MEM, label="Discharge memory"), Patch(color=C_MET, label="Meteorological (excl. soil moisture)"),
                        Patch(color=C_SM, label="Soil moisture (GWETTOP, GWETROOT, GWETPROF)"), Patch(color=C_SEAS, label="Seasonality")],
               loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.20), frameon=False)
    fig.tight_layout(); savefig(fig, "Fig_seasonal_shap_h7")

print("\nFigures ...")
draw_figures()

# ---------------------------------------------------------------------------
# 8.  Zip + download
# ---------------------------------------------------------------------------
el = time.time() - _T0
print(f"\nTotal runtime: {int(el // 3600)} h {int(el % 3600 // 60)} min {int(el % 60)} s")
sys.stdout = _ORIG_STDOUT; _LOG.close()
ZIP = OUT.rstrip("/") + ".zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, fn), fn)
print(f"Zip written: {ZIP}")
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    print("Not running in Colab — outputs are in", OUT)
